In [ ]:
import numpy as np
import pandas as pd
import pickle
import sys
import os
import torch
import torch.nn as nn

sys.path.append('..')
sys.path.append('../src')

from src.text_features import clean_text, decode_labels, TFIDFVectorizer, CombinedVectorizer, truncate_text, STYLE_DIM
from src.ffnn import FFNN


In [ ]:
df_subm = pd.read_csv('subm1.csv', sep=';')

texts = df_subm['Text'].tolist()

with open('../vectorizers/vectorizer_dnn.pkl', 'rb') as f:
    vec = pickle.load(f)

X_subm = vec.transform(texts)
X_subm_dense = X_subm.toarray() if not isinstance(X_subm, np.ndarray) else X_subm
X_tensor = torch.tensor(X_subm_dense, dtype=torch.float32)

print(f'Dados transformados. (shape: {X_tensor.shape})')


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_tensor = X_tensor.to(device)

input_dim = X_tensor.shape[1]
model_dnn = FFNN(input_dim=input_dim,
                 n_classes=5,
                 topology=[128, 64],
                 dropout=0.3).to(device)

model_dnn.load_state_dict(torch.load('../models/model_dnn.pt', map_location=device))
model_dnn.eval()

with torch.no_grad():
    outputs = model_dnn(X_tensor)
    preds_idx = torch.argmax(outputs, dim=1).cpu().numpy()


In [ ]:
preds_raw = decode_labels(preds_idx)

label_mapping = {
    'human': 'Human',
    'anthropic': 'Anthropic',
    'google': 'Google',
    'openai': 'OpenAI',
    'meta': 'Meta'
}

preds_formatted = [label_mapping.get(label.lower(), label) for label in preds_raw]

df_subm['Labels'] = preds_formatted
print(df_subm['Labels'].value_counts())
filename = 'subm1-g2-MEI-B.csv'
df_subm.to_csv(filename,index=False)

print(f"Primeiras linhas do ficheiro a submeter:\n{df_subm.head()}")
print(f"\nFicheiro '{filename}' guardado com sucesso!")